# Notebook 11: Ablation Studies

**R5 — Evaluation, Ablations and Drift Correction**

This notebook presents the full ablation study for the Drift-Sense VIO project.
All numbers are reproduced from pre-computed CSVs in `results/ablations/`.

In [ ]:
import sys
from pathlib import Path
root = Path('.').resolve().parent
sys.path.insert(0, str(root / 'src'))

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

RESULTS = root / 'results' / 'ablations'
FIGURES = root / 'figures' / 'dl'
FIGURES.mkdir(parents=True, exist_ok=True)
print('Root:', root)
pd.set_option('display.float_format', '{:.4f}'.format)


## Section 1: Phase 1 Comparability Check

We first verify that the new harness reproduces the Phase 1 benchmark numbers exactly.
The test suite (`tests/test_dl_eval.py`) already enforces:
- Mean baseline: 4.53 cm
- Lasso: 2.50 cm (MAE), R² = 0.76
- Physics-only: R² = 0.82

In [ ]:
from vio_drift.evaluation.dl_eval import reproduce_phase1_benchmarks
benchmarks = reproduce_phase1_benchmarks()
print('=== Phase 1 Benchmark Reproduction ===')
for name, metrics in benchmarks.items():
    print(f"  {name}: MAE={metrics['mae_cm']:.4f} cm, R²={metrics['r2']:.4f}")


## Section 2: Core Experiments E0–E7

Comparison of all model families against the Phase 1 Lasso baseline.
- **E0**: Phase 1 Lasso (anchor)
- **E1**: 1D-CNN (IMU-only)
- **E2**: LSTM (IMU-only)
- **E3**: Hybrid H1 (IMU-Net + hand-crafted + physics features)
- **E4**: Hybrid H2 (physics prior + residual network) ← **best**
- **E5**: Camera-only CNN
- **E6**: Hybrid H3 (IMU + Camera)
- **E7**: Best overall hybrid

In [ ]:
df_e = pd.read_csv(RESULTS / 'experiments_e0_e7.csv')
display_cols = ['experiment_id','name','family','mae_mean_cm','mae_std_cm','rmse_mean_cm','r2_mean','bias_mean_cm']
print(df_e[display_cols].to_string(index=False))


In [ ]:
LASSO_MAE = 2.5044
COL = {
    'Phase 1 (ML / Physics)': '#2196F3',
    'DL (IMU-only)': '#FF9800',
    'Hybrid': '#4CAF50',
}
fig, ax = plt.subplots(figsize=(12, 5))
colors = [COL.get(f, '#9E9E9E') for f in df_e['family']]
bars = ax.bar(df_e['experiment_id'], df_e['mae_mean_cm'],
              yerr=df_e['mae_std_cm'], capsize=4,
              color=colors, edgecolor='k', linewidth=0.6, zorder=3)
ax.axhline(LASSO_MAE, color='#E53935', linestyle='--', linewidth=1.4,
           label=f'Lasso baseline {LASSO_MAE:.2f} cm')
for bar, mae, std in zip(bars, df_e['mae_mean_cm'], df_e['mae_std_cm']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+std+0.4,
            f'{mae:.1f}', ha='center', va='bottom', fontsize=8)
patches = [mpatches.Patch(color=v, label=k) for k,v in COL.items()]
ax.legend(handles=patches+[plt.Line2D([0],[0],color='#E53935',linestyle='--',label=f'Lasso {LASSO_MAE:.2f} cm')], fontsize=8)
ax.set_xlabel('Experiment'); ax.set_ylabel('MAE (cm)')
ax.set_title('E0-E7: Experiment Comparison (MAE ± std, 3 seeds)')
ax.set_ylim(0, df_e['mae_mean_cm'].max()*1.25)
ax.grid(axis='y', alpha=0.4, zorder=0)
plt.tight_layout(); plt.show()
print(f"\nKey finding: E4 (Hybrid H2) MAE={df_e[df_e.experiment_id=='E4']['mae_mean_cm'].values[0]:.2f} cm < Lasso {LASSO_MAE:.2f} cm")


## Section 3: A1 — Regularisation Ablation

Effect of dropout, weight-decay, and early stopping on Hybrid H2.

In [ ]:
df_a1 = pd.read_csv(RESULTS / 'regularization_a1.csv')
print(df_a1[['variation','mae_mean_cm','mae_std_cm','rmse_mean_cm','r2_mean']].to_string(index=False))


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
x = range(len(df_a1))
ax.bar(x, df_a1['mae_mean_cm'], yerr=df_a1['mae_std_cm'], capsize=4,
       color='#26A69A', edgecolor='k', linewidth=0.6, zorder=3)
ax.set_xticks(list(x))
ax.set_xticklabels([str(v).replace('_',chr(10)) for v in df_a1['variation']], fontsize=9)
ax.set_ylabel('MAE (cm)'); ax.set_title('A1: Regularisation Ablation')
ax.grid(axis='y', alpha=0.4, zorder=0); plt.tight_layout(); plt.show()


## Section 4: A2 — Data Scaling

MAE and R² as a function of training data fraction (50%, 75%, 100%).

In [ ]:
df_a2 = pd.read_csv(RESULTS / 'data_scaling_a2.csv')
print(df_a2.to_string(index=False))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.errorbar(df_a2['fraction']*100, df_a2['mae_mean_cm'], yerr=df_a2['mae_std_cm'],
             marker='o', color='#1976D2', linewidth=1.8, capsize=4)
ax1.axhline(LASSO_MAE, color='#E53935', linestyle='--', label=f'Lasso {LASSO_MAE:.2f}')
ax1.set_xlabel('Training data (%)'); ax1.set_ylabel('MAE (cm)')
ax1.set_title('A2: MAE vs Training Size'); ax1.legend(fontsize=8); ax1.grid(alpha=0.4)
ax2.plot(df_a2['fraction']*100, df_a2['r2_mean'], marker='s', color='#388E3C', linewidth=1.8)
ax2.set_xlabel('Training data (%)'); ax2.set_ylabel('R²')
ax2.set_title('A2: R² vs Training Size'); ax2.grid(alpha=0.4)
plt.tight_layout(); plt.show()


## Section 5: A3 — Component Contributions

Absolving each component of the Hybrid H2 model (physics prior, residual net, etc.).

In [ ]:
df_a3 = pd.read_csv(RESULTS / 'components_a3.csv')
print(df_a3.to_string(index=False))


## Section 6: A4 — Seed Spread

MAE ± std across seeds 42, 123, 999 for all major models. Confirms reproducibility.

In [ ]:
df_a4 = pd.read_csv(RESULTS / 'seeds_a4.csv')
print(df_a4[['model_id','n_seeds','mae_mean_cm','mae_std_cm','mae_min_cm','mae_max_cm','r2_mean']].to_string(index=False))
import numpy as np
x = np.arange(len(df_a4))
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(x, df_a4['mae_mean_cm'], yerr=df_a4['mae_std_cm'], capsize=5,
       color='#5C6BC0', edgecolor='k', linewidth=0.6, zorder=3)
ax.axhline(LASSO_MAE, color='#E53935', linestyle='--', linewidth=1.4, label=f'Lasso {LASSO_MAE:.2f} cm')
for bar, mae, std in zip(ax.patches, df_a4['mae_mean_cm'], df_a4['mae_std_cm']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+std+0.3,
            f'{mae:.1f}±{std:.1f}', ha='center', va='bottom', fontsize=7)
ax.set_xticks(list(x)); ax.set_xticklabels(df_a4['model_id'])
ax.set_ylabel('MAE (cm)'); ax.set_title('A4: Seed Spread (3 seeds each)')
ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.4); plt.tight_layout(); plt.show()


## Section 7: A5 — Take-off vs Steady-state Segments

MAE split between the first 20% (take-off) and remaining 80% (steady flight) of each sequence.
Phase 1 required this segmented evaluation for comparability.

In [ ]:
df_a5 = pd.read_csv(RESULTS / 'segments_a5.csv')
print(df_a5.to_string(index=False))
seqs = df_a5['flight'].unique()
fig, axes = plt.subplots(1, len(seqs), figsize=(12, 5), sharey=False)
if len(seqs) == 1: axes = [axes]
for ax, seq in zip(axes, seqs):
    sub = df_a5[df_a5['flight']==seq]
    x2 = np.arange(len(sub)); w=0.35
    ax.bar(x2-w/2, sub['takeoff_mae_cm'], w, label='Take-off (first 20%)', color='#EF5350', edgecolor='k', linewidth=0.5)
    ax.bar(x2+w/2, sub['steady_mae_cm'],  w, label='Rest (80%)',           color='#42A5F5', edgecolor='k', linewidth=0.5)
    ax.set_xticks(list(x2)); ax.set_xticklabels(sub['model'], rotation=15, ha='right', fontsize=8)
    ax.set_title(seq); ax.set_ylabel('MAE (cm)'); ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.4)
fig.suptitle('A5: Take-off vs Steady-state MAE'); plt.tight_layout(); plt.show()


## Summary

| Experiment | MAE (cm) | R² | Note |
|---|---|---|---|
| E0 Lasso | 2.50 | 0.76 | Phase 1 baseline |
| E1 1D-CNN | 19.87 | -10.1 | Underfits small data |
| E2 LSTM | 4.36 | 0.33 | Moderate |
| E3 Hybrid H1 | 17.05 | -6.73 | — |
| **E4 Hybrid H2** | **2.30** | **0.79** | **Beats Lasso ✓** |
| E5 Camera | 16.45 | -8.30 | No real camera data |
| E6 IMU+Camera | 17.30 | -7.62 | — |
| E7 Best Hybrid | 2.41 | 0.78 | Ensemble variant |

**Key finding:** Hybrid H2 (physics prior + residual network) achieves 2.30 cm MAE, beating Lasso by 8% with comparable variance.